In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DateType,DoubleType
from pyspark.sql.functions import to_date
from datetime import date

schema=StructType([
StructField("id",IntegerType(),True),
StructField("name",StringType(),True),
StructField("salary",DoubleType(),True),
StructField("startDate",DateType(),True),
StructField("endDate",DateType(),True)
])

data=[(1,'Sanjay',123456,date(2026,6,4),date(2026,6,6)),(1,'Sanjay',123456,date(2026,6,4),date(2026,6,6)),(2,'SSS',None,date(2026,6,3),date(2026,6,6)),(1,'Sanjay',123456,date(2026,3,4),date(2026,4,6)),(4,'Sanjay arya  Ch',123456,date(2026,2,4),date(2026,1,6)),(5,'ROHIT  Rawat',123456,date(2026,4,4),date(2026,6,6))]

df=spark.createDataFrame(data,schema)
df.show()

In [0]:
# remove duplcates

df=df.dropDuplicates()

# #remove dup based on col
# df=df.dropDuplicates(["id"])

# # drop column
# df=df.drop("endDate")
df.show()


In [0]:
from pyspark.sql import functions as F
# df=df.withColumn("salary",F.coalesce(F.col("salary"),F.lit(0)))
df=df.na.fill({'salary': 12})
df.show()

In [0]:
## date converstion in UTC
df=df.withColumns({"startDate":F.to_timestamp(F.col("startDate")),
                   "endDate":F.to_timestamp(F.col("endDate"))})
df.show()

In [0]:
df=df.withColumn("name",F.initcap(F.trim(F.col("name"))))
df.show()

In [0]:
df=spark.read.format("json").option("multiLine",True).load("/Volumes/workspace/sanjay_volume/json_2026/sample_nested_1000_records.json")
df.display()

In [0]:
df_ex=df.withColumn("order",F.explode("orders")).withColumn("item",F.explode("order.items"))\
    .select("order.order_id","order.amount","item.product_id","item.quantity","address.city")
df_ex.display()

In [0]:
from pyspark.sql.functions import row_number,col,lead
from pyspark.sql.window import Window

win_spwe=Window.partitionBy("department").orderBy(col("salary").desc())

df_rn=df.withColumn("rn",row_number().over(win_spwe)).filter(col("rn")==1).drop("rn")
df_ld=df.withColumn("rn",lead(col("salary"),1).over(win_spwe))
df_rn.display()

In [0]:
%sql
with CTE as
(
select *,
row_number() over(partition by empid,department order by salary desc) as rn
from employee
)
delete from CTE where rn>1

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("xyz").getOrCreate()

from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType,DateType
from datetime import date
from pyspark.sql.functions import col,lit,coalesce,current_date
schema=StructType([
	StructField("id",IntegerType(),True),
	StructField("Name",StringType(),True),
	StructField("Salary",DoubleType(),True),
	StructField("StartDate",DateType(),True),
	StructField("EndDate",DateType(),True)])

Data=[(1,"Sanjay",77867,date(2026,4,3),date(2026,5,3)),(2,"Arya",774467,date(2026,4,18),date(2026,6,3)),(2,"Arya",774467,date(2026,4,18),date(2026,6,3)),(2,"RR",774467,date(2026,4,18),date(2026,6,3)),(3,"RR",774467,None,date(2026,6,3))]

df=spark.createDataFrame(Data,schema)
df.show()


In [0]:
dfd=df.dropDuplicates()
dfd.display()


In [0]:

dfd1=df.dropDuplicates(["id"])
dfd1.display()

In [0]:
##Nulls handle
from pyspark.sql.functions import col, lit, when

dfN1ls = df.withColumn(
    "StartDate",
    when(col("StartDate").isNull(), lit("2026-06-03").cast("date"))
    .otherwise(col("StartDate"))
)
dfN1ls.show()

In [0]:
dfcl=df.withColumn("StartDate",coalesce(col("StartDate"),lit(current_date())))
dfcl.show()

In [0]:
%sql
select * from delta. `/Volumes/workspace/sanjay_volume/2026/April/ofcdatata/`

In [0]:
%sql
CREATE TABLE sales_shallow DEEP CLONE delta. `/Volumes/workspace/sanjay_volume/2026/April/ofcdatata/`

In [0]:
%sql
select * from sales_shallow

In [0]:
%sql
CREATE TABLE sales_deep    DEEP CLONE delta. `/Volumes/workspace/sanjay_volume/2026/April/ofcdatata/`

In [0]:
%sql
select * from sales_deep